In [7]:
from pathlib import Path
import math, os, json, random
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoFeatureExtractor, WavLMModel

# =====================
# Paths
# =====================
DATA_DIR  = Path("/home/danila/networks/data")
AUDIO_DIR = DATA_DIR / "audio"          # mp3: 00000.mp3 ...
TRAIN_CSV = DATA_DIR / "train_split.csv"
VALID_CSV = DATA_DIR / "valid_split.csv"

OUT_DIR = DATA_DIR / "runs" / "audio_wavlm_end2end_fps5_v1"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH = OUT_DIR / "best_by_val_pearson.pt"
HIST_PATH = OUT_DIR / "history.json"

# =====================
# Task
# =====================
EMOTIONS = ["Admiration","Amusement","Determination","Empathic Pain","Excitement","Joy"]
NUM_TARGETS = len(EMOTIONS)
ID_WIDTH = 5

# =====================
# Audio
# =====================
TARGET_SR = 16_000

# Crop control (IMPORTANT):
AUDIO_MAX_SEC = 20.0        # crop to first N seconds
CROP_MODE = "first"         # "first" or "random"

# fps binning (to match your previous pipeline)
TARGET_FPS = 5
HOP_SEC = 1.0 / TARGET_FPS  # 0.2 sec
NUM_BINS = int(math.ceil(AUDIO_MAX_SEC / HOP_SEC))  # e.g. 12s -> 60 bins

# =====================
# Model
# =====================
MODEL_NAME = "microsoft/wavlm-large"

# =====================
# Training
# =====================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42

BATCH_SIZE = 4                 # small due to WavLM-Large
GRAD_ACCUM_STEPS = 8           # effective batch = 32
EPOCHS = 40
LR_HEAD = 1e-4
LR_ENCODER = 1e-5
WEIGHT_DECAY_HEAD = 1e-2
WEIGHT_DECAY_ENCODER = 0.0 
NUM_WARMUP_STEPS = 0

USE_AMP = (DEVICE == "cuda")
GRAD_CLIP = 1.0

# Loss:
LOSS_MODE = "mse"              # "mse" or "mse+pearson"
ALPHA_PEARSON = 0.2            # used if LOSS_MODE == "mse+pearson"

# Memory tricks
USE_GRAD_CHECKPOINTING = False

torch.backends.cuda.matmul.allow_tf32 = True
print("Device:", DEVICE, "| bins:", NUM_BINS)

Device: cuda | bins: 100


In [8]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

In [9]:
train_df = pd.read_csv(TRAIN_CSV, dtype={"Filename": str})
valid_df = pd.read_csv(VALID_CSV, dtype={"Filename": str})

train_df["Filename"] = train_df["Filename"].str.zfill(ID_WIDTH)
valid_df["Filename"] = valid_df["Filename"].str.zfill(ID_WIDTH)

print("Train:", len(train_df), "Val:", len(valid_df))

# target normalization on TRAIN (Pearson invariant to affine scaling)
y_train = train_df[EMOTIONS].values.astype(np.float32)
y_mean = y_train.mean(axis=0)
y_std  = np.clip(y_train.std(axis=0), 1e-3, None)

y_mean, y_std

Train: 8072 Val: 4588


(array([0.17847925, 0.1253244 , 0.09674559, 0.02144948, 0.14203759,
        0.11574291], dtype=float32),
 array([0.24964707, 0.21797715, 0.18756332, 0.09568307, 0.22837305,
        0.21251066], dtype=float32))

In [10]:
def load_audio_mono_16k(mp3_path: Path, target_sr: int = 16000):
    """
    Returns:
      wav: torch.FloatTensor [T] ~[-1, 1]
      sr: int
    """
    try:
        import torchaudio
        wav, sr = torchaudio.load(str(mp3_path))  # [C, T]
        if wav.ndim == 2 and wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
        wav = wav.squeeze(0)  # [T]

        if sr != target_sr:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=target_sr)
            wav = resampler(wav.unsqueeze(0)).squeeze(0)
            sr = target_sr
        return wav.contiguous().float(), sr

    except Exception:
        import librosa
        x, sr = librosa.load(str(mp3_path), sr=target_sr, mono=True)
        wav = torch.from_numpy(x).float()
        return wav.contiguous(), target_sr


def crop_wav(wav: torch.Tensor, sr: int, max_sec: float, mode: str = "first") -> torch.Tensor:
    if max_sec is None or max_sec <= 0:
        return wav
    max_samples = int(max_sec * sr)
    if wav.numel() <= max_samples:
        return wav
    if mode == "first":
        return wav[:max_samples]
    if mode == "random":
        start = random.randint(0, wav.numel() - max_samples)
        return wav[start:start + max_samples]
    raise ValueError(f"Unknown CROP_MODE={mode}")

In [11]:
class AudioWavDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True)

        keep = []
        self.missing = []
        for i, row in self.df.iterrows():
            vid = str(row["Filename"]).zfill(ID_WIDTH)
            p = AUDIO_DIR / f"{vid}.mp3"
            if p.exists():
                keep.append(i)
            else:
                self.missing.append(vid)

        self.df = self.df.iloc[keep].reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        vid = str(row["Filename"]).zfill(ID_WIDTH)
        mp3 = AUDIO_DIR / f"{vid}.mp3"

        wav, sr = load_audio_mono_16k(mp3, TARGET_SR)
        wav = crop_wav(wav, sr, AUDIO_MAX_SEC, CROP_MODE)

        y = np.array([row[e] for e in EMOTIONS], dtype=np.float32)
        y = (y - y_mean) / y_std

        return {"video_id": vid, "wav": wav, "sr": sr, "y": y}


train_ds = AudioWavDataset(train_df)
valid_ds = AudioWavDataset(valid_df)

print("Train usable:", len(train_ds), "missing mp3:", len(train_ds.missing))
print("Val   usable:", len(valid_ds), "missing mp3:", len(valid_ds.missing))

Train usable: 8072 missing mp3: 0
Val   usable: 4588 missing mp3: 0


In [12]:
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)

def collate_audio(batch):
    wavs = [b["wav"].numpy() for b in batch]
    ys = torch.tensor([b["y"] for b in batch], dtype=torch.float32)
    vids = [b["video_id"] for b in batch]

    fe = feature_extractor(
        wavs,
        sampling_rate=TARGET_SR,
        padding=True,
        return_attention_mask=True,
        return_tensors="pt",
    )
    return {"video_id": vids, "fe": fe, "y": ys}

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, collate_fn=collate_audio)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_audio)

print("Batches | train:", len(train_loader), "val:", len(valid_loader))

Batches | train: 2018 val: 1147


In [13]:
def time_mask_bins(x, m, p=0.3, max_ratio=0.15):
    # x: [B,T,D], m: [B,T] bool
    if (not x.requires_grad) and (not x.is_cuda):
        pass
    if random.random() > p:
        return x
    B, T, D = x.shape
    for b in range(B):
        Tb = int(m[b].sum().item())
        if Tb < 4:
            continue
        w = max(1, int(Tb * max_ratio))
        s = random.randint(0, max(0, Tb - w))
        x[b, s:s+w, :] = 0
    return x

In [14]:
class TCNBlock(nn.Module):
    def __init__(self, d: int, kernel: int, dilation: int, dropout: float):
        super().__init__()
        padding = (kernel - 1) * dilation // 2
        self.conv1 = nn.Conv1d(d, d, kernel_size=kernel, dilation=dilation, padding=padding)
        self.conv2 = nn.Conv1d(d, d, kernel_size=kernel, dilation=dilation, padding=padding)
        self.norm1 = nn.GroupNorm(1, d)
        self.norm2 = nn.GroupNorm(1, d)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        res = x
        x = self.drop(F.gelu(self.norm1(self.conv1(x))))
        x = self.drop(F.gelu(self.norm2(self.conv2(x))))
        return x + res

class TCNEncoder(nn.Module):
    def __init__(self, d: int, layers: int, kernel: int, dropout: float):
        super().__init__()
        self.blocks = nn.ModuleList([TCNBlock(d, kernel, 2**i, dropout) for i in range(layers)])

    def forward(self, x):  # [B,T,d]
        x = x.transpose(1, 2)  # [B,d,T]
        for b in self.blocks:
            x = b(x)
        return x.transpose(1, 2)

class AttentiveStatsPooling(nn.Module):
    def __init__(self, d: int, attn_hidden: int, dropout: float, temp: float = 1.5):
        super().__init__()
        self.temp = temp
        self.attn = nn.Sequential(
            nn.Linear(d, attn_hidden),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(attn_hidden, 1),
        )

    def forward(self, x, mask):  # x: [B,T,d], mask: [B,T]
        logits = self.attn(x).squeeze(-1) / self.temp
        logits = logits.masked_fill(~mask, -1e4)

        w = torch.softmax(logits, dim=1)
        w = w * mask.float()
        w = w / (w.sum(dim=1, keepdim=True) + 1e-6)
        w = w.unsqueeze(-1)

        mu = (w * x).sum(dim=1)
        var = (w * (x - mu.unsqueeze(1)).pow(2)).sum(dim=1)
        std = torch.sqrt(var + 1e-6)
        return torch.cat([mu, std], dim=-1)

def pearson_corr_torch(preds: torch.Tensor, targets: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    vx = preds - preds.mean(0)
    vy = targets - targets.mean(0)
    corr = (vx * vy).sum(0) / (torch.sqrt((vx**2).sum(0) * (vy**2).sum(0)) + eps)
    return corr.mean()

def hidden_to_fps_bins(hidden: torch.Tensor,
                       input_lengths: torch.Tensor,
                       output_lengths: torch.Tensor,
                       sr: int,
                       hop_sec: float,
                       num_bins: int):
    """
    hidden: [B, Lmax, H] (padded to max in batch)
    input_lengths: [B] number of valid input samples (from attention_mask.sum)
    output_lengths: [B] number of valid hidden frames (from _get_feat_extract_output_lengths)
    """
    B, Lmax, H = hidden.shape
    device = hidden.device

    dur = input_lengths.float() / float(sr)  # [B]

    x_out, m_out = [], []
    for b in range(B):
        Lb = int(output_lengths[b].item())
        if Lb <= 0:
            x_out.append(torch.zeros((num_bins, H), device=device, dtype=hidden.dtype))
            m_out.append(torch.zeros((num_bins,), device=device, dtype=torch.bool))
            continue

        hb = hidden[b, :Lb, :]  # valid frames only
        stride = dur[b] / float(Lb)  # seconds per hidden frame

        t = torch.arange(Lb, device=device, dtype=torch.float32) * stride
        idx = torch.floor(t / float(hop_sec)).long()
        valid = (idx >= 0) & (idx < num_bins)

        sums = torch.zeros((num_bins, H), device=device, dtype=hidden.dtype)
        cnts = torch.zeros((num_bins,), device=device, dtype=torch.int32)

        if valid.any():
            ii = idx[valid]
            sums.index_add_(0, ii, hb[valid])
            cnts.index_add_(0, ii, torch.ones_like(ii, dtype=torch.int32))

        xb = sums / cnts.clamp_min(1).unsqueeze(1).to(sums.dtype)
        mb = cnts > 0

        x_out.append(xb)
        m_out.append(mb)

    return torch.stack(x_out, dim=0), torch.stack(m_out, dim=0)

class AudioWavLMEnd2End(nn.Module):
    def __init__(self, encoder: WavLMModel, d_model: int = 192, tcn_layers: int = 6, tcn_kernel: int = 3,
                 dropout: float = 0.3, attn_hidden: int = 128):
        super().__init__()
        self.encoder = encoder
        h_in = int(encoder.config.hidden_size)

        self.proj = nn.Sequential(
            nn.Linear(h_in, d_model),
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
        )
        self.tcn = TCNEncoder(d_model, tcn_layers, tcn_kernel, dropout)
        self.pool = AttentiveStatsPooling(d_model, attn_hidden, dropout, temp=1.5)
        self.head = nn.Sequential(
            nn.Linear(2*d_model, 2*d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(2*d_model, NUM_TARGETS),
        )

    def forward(self, fe):
        input_values = fe["input_values"]  # [B, T]
        attention_mask = fe.get("attention_mask", None)
        if attention_mask is None:
            attention_mask = torch.ones_like(input_values, dtype=torch.long)
    
        input_lengths = attention_mask.sum(-1)  # [B] in samples after feature_extractor padding
    
        # KEY: the valid length is hidden after the conv feature extractor
        output_lengths = self.encoder._get_feat_extract_output_lengths(input_lengths)
    
        out = self.encoder(input_values=input_values, attention_mask=attention_mask)
        hidden = out.last_hidden_state  # [B, Lmax, H]
    
        x_bins, m_bins = hidden_to_fps_bins(
            hidden=hidden,
            input_lengths=input_lengths,
            output_lengths=output_lengths,
            sr=TARGET_SR,
            hop_sec=HOP_SEC,
            num_bins=NUM_BINS,
        )
        if self.training:
            x_bins = time_mask_bins(x_bins, m_bins, p=0.3, max_ratio=0.15)
        
        x = self.proj(x_bins)
        x = self.tcn(x)
        z = self.pool(x, m_bins)
        return self.head(z)

encoder = WavLMModel.from_pretrained(MODEL_NAME)
if USE_GRAD_CHECKPOINTING:
    encoder.gradient_checkpointing_enable()

model = AudioWavLMEnd2End(encoder).to(DEVICE)
print("WavLM hidden:", encoder.config.hidden_size)

WavLM hidden: 1024


In [15]:
mse_loss = nn.MSELoss()

def loss_fn(pred, y):
    if LOSS_MODE == "mse":
        return mse_loss(pred, y)
    if LOSS_MODE == "mse+pearson":
        return mse_loss(pred, y) + ALPHA_PEARSON * (1.0 - pearson_corr_torch(pred, y))
    raise ValueError(LOSS_MODE)

In [16]:
enc_params = model.encoder.parameters()
head_params = list(model.proj.parameters()) + list(model.tcn.parameters()) + list(model.pool.parameters()) + list(model.head.parameters())

optimizer = torch.optim.AdamW(
    [
        {"params": enc_params, "lr": LR_ENCODER, "weight_decay": WEIGHT_DECAY_ENCODER},
        {"params": head_params, "lr": LR_HEAD, "weight_decay": WEIGHT_DECAY_HEAD},
    ]
)

total_steps = EPOCHS * math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
warmup_steps = max(0, int(NUM_WARMUP_STEPS))

def lr_lambda(step):
    if warmup_steps > 0 and step < warmup_steps:
        return float(step) / float(max(1, warmup_steps))
    denom = max(1, total_steps - warmup_steps)
    progress = (step - warmup_steps) / float(denom)
    return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

print("Total optimizer steps:", total_steps, "warmup:", warmup_steps)

Total optimizer steps: 10120 warmup: 0


/tmp/ipykernel_189495/4222639651.py:22: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


In [17]:
def move_fe_to_device(fe, device):
    return {k: v.to(device, non_blocking=True) for k, v in fe.items()}

@torch.no_grad()
def eval_epoch(model, loader):
    model.eval()
    Ps, Ys = [], []
    total_loss = 0.0
    n = 0

    for batch in tqdm(loader, desc="eval", leave=False):
        fe = move_fe_to_device(batch["fe"], DEVICE)
        y = batch["y"].to(DEVICE, non_blocking=True)

        pred = model(fe)
        loss = loss_fn(pred, y)

        total_loss += loss.item() * y.size(0)
        n += y.size(0)
        Ps.append(pred.detach().float().cpu())
        Ys.append(y.detach().float().cpu())

    P = torch.cat(Ps, dim=0)
    Y = torch.cat(Ys, dim=0)
    corr = float(pearson_corr_torch(P, Y).item())
    return {"loss": total_loss / max(1, n), "corr": corr}

def train_epoch(model, loader):
    model.train()
    total_loss = 0.0
    n = 0

    optimizer.zero_grad(set_to_none=True)
    step_idx = 0

    for it, batch in enumerate(tqdm(loader, desc="train", leave=False), start=1):
        fe = move_fe_to_device(batch["fe"], DEVICE)
        y = batch["y"].to(DEVICE, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            pred = model(fe)
            loss = loss_fn(pred, y) / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()

        total_loss += loss.item() * y.size(0) * GRAD_ACCUM_STEPS
        n += y.size(0)

        do_step = (it % GRAD_ACCUM_STEPS == 0) or (it == len(loader))
        if do_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            step_idx += 1

    return {"loss": total_loss / max(1, n), "steps": step_idx}

In [ ]:
history = []
best_corr = -1e9
best_epoch = -1
patience = 10
bad = 0

for epoch in range(1, EPOCHS + 1):
    tr = train_epoch(model, train_loader)
    va = eval_epoch(model, valid_loader)

    lr_now = optimizer.param_groups[0]["lr"]
    row = {
        "epoch": epoch,
        "train_loss": float(tr["loss"]),
        "val_loss": float(va["loss"]),
        "val_corr": float(va["corr"]),
        "lr": float(lr_now),
    }
    history.append(row)

    print(f"Epoch {epoch:03d} | train_loss {tr['loss']:.4f} | val_loss {va['loss']:.4f} | val_mean_pearson {va['corr']:.4f} | lr {lr_now:.2e}")

    with open(HIST_PATH, "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)

    if va["corr"] > best_corr + 1e-4:
        best_corr = va["corr"]
        best_epoch = epoch
        bad = 0
        torch.save({"epoch": epoch, "model_state": model.state_dict(), "best_corr": best_corr}, CKPT_PATH)
        print(f"✅ Saved best: epoch={epoch}, val_corr={best_corr:.4f}")
    else:
        bad += 1
        print(f"⏳ No improvement: {bad}/{patience}")

    if bad >= patience:
        print(f"🛑 Early stopping. Best epoch={best_epoch}, best val corr={best_corr:.4f}")
        break

print("Done. Best:", best_corr, "at epoch", best_epoch)
print("Checkpoint:", CKPT_PATH)

train:   0%|          | 0/2018 [00:00<?, ?it/s]

/tmp/ipykernel_175177/3800485816.py:5: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  ys = torch.tensor([b["y"] for b in batch], dtype=torch.float32)
/tmp/ipykernel_175177/2403714594.py:40: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):
/home/danila/networks/.venv/lib/python3.12/site-packages/torch/nn/functional.py:6044: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


eval:   0%|          | 0/1147 [00:00<?, ?it/s]

Epoch 001 | train_loss 1.0054 | val_loss 0.8249 | val_mean_pearson 0.1955 | lr 9.98e-06
✅ Saved best: epoch=1, val_corr=0.1955


train:   0%|          | 0/2018 [00:00<?, ?it/s]

...

Epoch 007 | train_loss 0.7802 | val_loss 0.7501 | val_mean_pearson 0.4077 | lr 9.26e-06
✅ Saved best: epoch=7, val_corr=0.4077


train:   0%|          | 0/2018 [00:00<?, ?it/s]

eval:   0%|          | 0/1147 [00:00<?, ?it/s]

Epoch 008 | train_loss 0.7574 | val_loss 0.7492 | val_mean_pearson 0.4127 | lr 9.05e-06
✅ Saved best: epoch=8, val_corr=0.4127


train:   0%|          | 0/2018 [00:00<?, ?it/s]

In [21]:
ckpt = torch.load(CKPT_PATH, map_location="cpu")
model.load_state_dict(ckpt["model_state"])

model.encoder.save_pretrained(OUT_DIR/"best_encoder")

model.to(DEVICE)

va = eval_epoch(model, valid_loader)
print("Best checkpoint epoch:", ckpt["epoch"])
print("Val loss:", va["loss"])
print("Val mean Pearson:", va["corr"])

eval:   0%|          | 0/1147 [00:00<?, ?it/s]

/tmp/ipykernel_186598/3800485816.py:5: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  ys = torch.tensor([b["y"] for b in batch], dtype=torch.float32)
/home/danila/networks/.venv/lib/python3.12/site-packages/torch/nn/functional.py:6044: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


Best checkpoint epoch: 10
Val loss: 0.7549438276981777
Val mean Pearson: 0.4174387454986572


### per emotion correlation

In [22]:
ckpt = torch.load(CKPT_PATH, map_location="cpu")
model.load_state_dict(ckpt["model_state"])

model.encoder.save_pretrained(OUT_DIR/"best_encoder")

model.to(DEVICE)

va = eval_epoch(model, valid_loader)
print("Best checkpoint epoch:", ckpt["epoch"])
print("Val loss:", va["loss"])
print("Val mean Pearson:", va["corr"])

# --- per-emotion Pearson on val ---
@torch.no_grad()
def pearson_per_emotion(P: torch.Tensor, Y: torch.Tensor, eps: float = 1e-8):
    """
    P, Y: [N, 6] on CPU or GPU
    returns: (mean_corr: float, per_corr: list[float])
    """
    P = P.float()
    Y = Y.float()
    vx = P - P.mean(dim=0, keepdim=True)
    vy = Y - Y.mean(dim=0, keepdim=True)
    num = (vx * vy).sum(dim=0)
    den = torch.sqrt((vx * vx).sum(dim=0) * (vy * vy).sum(dim=0)) + eps
    corr = num / den
    return float(corr.mean().item()), [float(c.item()) for c in corr]

@torch.no_grad()
def eval_epoch_with_per_emotion(model, loader):
    model.eval()
    Ps, Ys = [], []
    for batch in tqdm(loader, desc="eval(per-emotion)", leave=False):
        fe = {k: v.to(DEVICE, non_blocking=True) for k, v in batch["fe"].items()}
        y  = batch["y"].to(DEVICE, non_blocking=True)
        p  = model(fe)
        Ps.append(p.detach().float().cpu())
        Ys.append(y.detach().float().cpu())
    P = torch.cat(Ps, dim=0)
    Y = torch.cat(Ys, dim=0)
    mean_corr, per_corr = pearson_per_emotion(P, Y)
    return {"mean": mean_corr, "per": per_corr}

pe = eval_epoch_with_per_emotion(model, valid_loader)
print("Val mean Pearson (recomputed):", pe["mean"])
for name, c in zip(EMOTIONS, pe["per"]):
    print(f"  - {name:14s}: {c:.4f}")

eval:   0%|          | 0/1147 [00:00<?, ?it/s]

Best checkpoint epoch: 10
Val loss: 0.7549439653975089
Val mean Pearson: 0.41743865609169006


eval(per-emotion):   0%|          | 0/1147 [00:00<?, ?it/s]

Val mean Pearson (recomputed): 0.417438268661499
  - Admiration    : 0.4660
  - Amusement     : 0.4049
  - Determination : 0.3638
  - Empathic Pain : 0.5267
  - Excitement    : 0.3811
  - Joy           : 0.3622


### save best encoder

In [20]:
ckpt = torch.load(CKPT_PATH, map_location="cpu")
model.load_state_dict(ckpt["model_state"])
model.to(DEVICE)

va = eval_epoch(model, valid_loader)
print("Best checkpoint epoch:", ckpt["epoch"])
print("Val loss:", va["loss"])
print("Val mean Pearson:", va["corr"])

# =====================
# Save BEST encoder separately (after loading best weights)
# =====================
ENCODER_SAVE_DIR = OUT_DIR / "best_encoder"
ENCODER_SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Save encoder in HF format
model.encoder.save_pretrained(ENCODER_SAVE_DIR)

# save feature extractor too for consistent preprocessing
feature_extractor.save_pretrained(ENCODER_SAVE_DIR)

# Save run config (handy for reproducibility)
with open(ENCODER_SAVE_DIR / "run_config.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "model_name": MODEL_NAME,
            "audio_max_sec": AUDIO_MAX_SEC,
            "crop_mode": CROP_MODE,
            "target_sr": TARGET_SR,
            "target_fps": TARGET_FPS,
            "hop_sec": HOP_SEC,
            "num_bins": NUM_BINS,
            "emotions": EMOTIONS,
            "target_norm": {"y_mean": y_mean.tolist(), "y_std": y_std.tolist()},
            "best_epoch": int(ckpt.get("epoch", -1)),
            "best_val_corr": float(ckpt.get("best_corr", float("nan"))),
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

print("✅ Saved best encoder (+ feature_extractor) to:", ENCODER_SAVE_DIR)

eval:   0%|          | 0/1147 [00:00<?, ?it/s]

/tmp/ipykernel_189495/3800485816.py:5: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  ys = torch.tensor([b["y"] for b in batch], dtype=torch.float32)
/home/danila/networks/.venv/lib/python3.12/site-packages/torch/nn/functional.py:6044: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


Best checkpoint epoch: 10
Val loss: 0.75494421573247
Val mean Pearson: 0.417438268661499
✅ Saved best encoder (+ feature_extractor) to: /home/danila/networks/data/runs/audio_wavlm_end2end_fps5_v1/best_encoder


In [22]:
# =====================
# Save finetuned encoder + NEW head (for embeddings-based inference later)
# Run AFTER training, after loading best checkpoint into `model`
# =====================

from pathlib import Path
import json
import torch

ENC_FT_DIR = OUT_DIR / "finetuned_encoder"
ENC_FT_DIR.mkdir(parents=True, exist_ok=True)

HEAD_FT_DIR = OUT_DIR / "finetuned_head"
HEAD_FT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Save encoder (HF format) ----
model.encoder.save_pretrained(ENC_FT_DIR)
feature_extractor.save_pretrained(ENC_FT_DIR)

with open(ENC_FT_DIR / "meta.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "base_model": MODEL_NAME,
            "checkpoint_loaded": str(CKPT_PATH),
            "best_epoch": int(ckpt.get("epoch", -1)),
            "best_val_corr": float(ckpt.get("best_corr", float("nan"))),
            "audio_max_sec": float(AUDIO_MAX_SEC),
            "crop_mode": CROP_MODE,
            "target_sr": int(TARGET_SR),
            "target_fps": int(TARGET_FPS),
            "hop_sec": float(HOP_SEC),
            "num_bins": int(NUM_BINS),
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

print("✅ Saved finetuned encoder (+ feature_extractor) to:", ENC_FT_DIR)

# ---- Save NEW head weights (proj + TCN + pooling + head) ----
# We export head state dict in naming compatible with AudioWavLMModel:
#   proj.*, enc.*, pool.*, head.*
full_sd = model.state_dict()

head_sd = {}
for k, v in full_sd.items():
    if k.startswith("encoder."):
        continue

    # rename tcn.* -> enc.* for compatibility with AudioWavLMModel
    if k.startswith("tcn."):
        k2 = "enc." + k[len("tcn."):]
    else:
        k2 = k

    head_sd[k2] = v.detach().cpu()

# binary package (weights live here)
head_pkg = {
    "head_state": head_sd,                           # <-- tensors OK in torch.save
    "h_in": int(model.encoder.config.hidden_size),
    "num_targets": int(NUM_TARGETS),
    "audio_max_sec": float(AUDIO_MAX_SEC),
    "target_sr": int(TARGET_SR),
    "target_fps": int(TARGET_FPS),
    "hop_sec": float(HOP_SEC),
    "num_bins": int(NUM_BINS),
}

torch.save(head_pkg, HEAD_FT_DIR / "head.pt")

# JSON meta (NO tensors!)
head_meta = {k: v for k, v in head_pkg.items() if k != "head_state"}
head_meta["num_params"] = int(sum(p.numel() for p in head_sd.values()))
head_meta["state_keys"] = sorted(list(head_sd.keys()))[:50]  # первые 50 ключей для дебага
head_meta["state_keys_total"] = int(len(head_sd))

(HEAD_FT_DIR / "meta.json").write_text(
    json.dumps(head_meta, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("✅ Saved finetuned head to:", HEAD_FT_DIR / "head.pt")
print("✅ Saved head meta to   :", HEAD_FT_DIR / "meta.json")
print("Head keys example:", head_meta["state_keys"][:5])

✅ Saved finetuned encoder (+ feature_extractor) to: /home/danila/networks/data/runs/audio_wavlm_end2end_fps5_v1/finetuned_encoder
✅ Saved finetuned head to: /home/danila/networks/data/runs/audio_wavlm_end2end_fps5_v1/finetuned_head/head.pt
✅ Saved head meta to   : /home/danila/networks/data/runs/audio_wavlm_end2end_fps5_v1/finetuned_head/meta.json
Head keys example: ['enc.blocks.0.conv1.bias', 'enc.blocks.0.conv1.weight', 'enc.blocks.0.conv2.bias', 'enc.blocks.0.conv2.weight', 'enc.blocks.0.norm1.bias']


# Recalculating embeggings with finetuend encoder

In [1]:
from pathlib import Path
import math, json, traceback
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

# -----------------
# Where your mp3 live
# -----------------
DATA_DIR  = Path("/home/danila/networks/data")
AUDIO_DIR = DATA_DIR / "audio"
TRAIN_CSV = DATA_DIR / "train_split.csv"
VALID_CSV = DATA_DIR / "valid_split.csv"

# -----------------
# Load finetuned encoder
# -----------------
ENC_FT_DIR = DATA_DIR / "runs" / "audio_wavlm_end2end_fps5_v1" / "finetuned_encoder"

# -----------------
# Extraction settings
# -----------------
TARGET_SR = 16_000
TARGET_FPS = 5
HOP_SEC = 1.0 / TARGET_FPS

AUDIO_MAX_SEC = 20.0
CROP_MODE = "first"

# Fixed bins length for all samples
NUM_BINS = int(math.ceil(AUDIO_MAX_SEC / HOP_SEC))

# Chunking
CHUNK_SEC = 20.0
OVERLAP_SEC = 2.0

# Runtime
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = (DEVICE == "cuda")

# Saving
OUT_EMB_DIR = DATA_DIR / "embeddings" / f"audio_wavlm_large_fps{TARGET_FPS}_finetuned_sec{int(AUDIO_MAX_SEC)}_v1"
OUT_EMB_DIR.mkdir(parents=True, exist_ok=True)
INDEX_PATH = OUT_EMB_DIR / "index.parquet"

EMB_SAVE_DTYPE = "float16"  # "float16" or "float32"
SKIP_IF_EXISTS = True
ID_WIDTH = 5

def _ts():
    return datetime.now().strftime("%H:%M:%S")

print("ENC_FT_DIR:", ENC_FT_DIR)
print("OUT_EMB_DIR:", OUT_EMB_DIR)
print("NUM_BINS:", NUM_BINS, "HOP_SEC:", HOP_SEC)

ENC_FT_DIR: /home/danila/networks/data/runs/audio_wavlm_end2end_fps5_v1/finetuned_encoder
OUT_EMB_DIR: /home/danila/networks/data/embeddings/audio_wavlm_large_fps5_finetuned_sec20_v1
NUM_BINS: 100 HOP_SEC: 0.2


In [2]:
import random
from transformers import AutoFeatureExtractor, WavLMModel

# ids from splits
train_df = pd.read_csv(TRAIN_CSV, dtype={"Filename": str})
valid_df = pd.read_csv(VALID_CSV, dtype={"Filename": str})
train_df["Filename"] = train_df["Filename"].str.zfill(ID_WIDTH)
valid_df["Filename"] = valid_df["Filename"].str.zfill(ID_WIDTH)
all_ids = sorted(pd.concat([train_df["Filename"], valid_df["Filename"]]).unique().tolist())

tqdm.write(f"[{_ts()}] Total ids: {len(all_ids)} | examples: {all_ids[:10]}")

# load finetuned encoder + feature extractor
feature_extractor = AutoFeatureExtractor.from_pretrained(ENC_FT_DIR)
encoder = WavLMModel.from_pretrained(ENC_FT_DIR).to(DEVICE).eval()

HIDDEN = int(encoder.config.hidden_size)
tqdm.write(f"[{_ts()}] Loaded finetuned encoder: {ENC_FT_DIR} | hidden={HIDDEN} | device={DEVICE}")

[17:55:34] Total ids: 12660 | examples: ['00000', '00001', '00002', '00003', '00004', '00005', '00006', '00007', '00008', '00009']
[17:55:35] Loaded finetuned encoder: /home/danila/networks/data/runs/audio_wavlm_end2end_fps5_v1/finetuned_encoder | hidden=1024 | device=cuda


In [3]:
import random
from transformers import AutoFeatureExtractor, WavLMModel

# ids from splits
train_df = pd.read_csv(TRAIN_CSV, dtype={"Filename": str})
valid_df = pd.read_csv(VALID_CSV, dtype={"Filename": str})
train_df["Filename"] = train_df["Filename"].str.zfill(ID_WIDTH)
valid_df["Filename"] = valid_df["Filename"].str.zfill(ID_WIDTH)
all_ids = sorted(pd.concat([train_df["Filename"], valid_df["Filename"]]).unique().tolist())

tqdm.write(f"[{_ts()}] Total ids: {len(all_ids)} | examples: {all_ids[:10]}")

# load finetuned encoder + feature extractor
feature_extractor = AutoFeatureExtractor.from_pretrained(ENC_FT_DIR)
encoder = WavLMModel.from_pretrained(ENC_FT_DIR).to(DEVICE).eval()

HIDDEN = int(encoder.config.hidden_size)
tqdm.write(f"[{_ts()}] Loaded finetuned encoder: {ENC_FT_DIR} | hidden={HIDDEN} | device={DEVICE}")

[17:55:35] Total ids: 12660 | examples: ['00000', '00001', '00002', '00003', '00004', '00005', '00006', '00007', '00008', '00009']
[17:55:36] Loaded finetuned encoder: /home/danila/networks/data/runs/audio_wavlm_end2end_fps5_v1/finetuned_encoder | hidden=1024 | device=cuda


In [4]:
def load_audio_mono_16k(mp3_path: Path, target_sr: int = 16000):
    """
    Returns wav: torch.FloatTensor [T] mono, sr=target_sr
    """
    try:
        import torchaudio
        wav, sr = torchaudio.load(str(mp3_path))  # [C,T]
        if wav.ndim == 2 and wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
        wav = wav.squeeze(0)  # [T]
        if sr != target_sr:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=target_sr)
            wav = resampler(wav.unsqueeze(0)).squeeze(0)
            sr = target_sr
        return wav.contiguous().float(), sr
    except Exception as e_ta:
        try:
            import librosa
            x, _ = librosa.load(str(mp3_path), sr=target_sr, mono=True)
            return torch.from_numpy(x).float().contiguous(), target_sr
        except Exception as e_lr:
            raise RuntimeError(f"Failed to load audio ({e_ta}) / ({e_lr})")

def crop_wav(wav: torch.Tensor, sr: int, max_sec: float, mode: str = "first") -> torch.Tensor:
    if max_sec is None or max_sec <= 0:
        return wav
    max_samples = int(max_sec * sr)
    if wav.numel() <= max_samples:
        return wav
    if mode == "first":
        return wav[:max_samples]
    if mode == "random":
        start = random.randint(0, wav.numel() - max_samples)
        return wav[start:start + max_samples]
    raise ValueError(mode)

@torch.inference_mode()
def wavlm_hidden_chunk(wav_1d: torch.Tensor, sr: int):
    """
    wav_1d: [T] CPU float32 (one chunk)
    returns:
      hidden: [L, H] on DEVICE
      stride_sec: seconds per hidden frame
    """
    fe = feature_extractor(
        wav_1d.numpy(),
        sampling_rate=sr,
        return_tensors="pt",
        padding=False,
        return_attention_mask=True,
    )
    fe = {k: v.to(DEVICE) for k, v in fe.items()}

    input_values = fe["input_values"]          # [1, Tin]
    attention_mask = fe.get("attention_mask")  # [1, Tin]

    # valid input length in samples
    in_len = attention_mask.sum(-1)  # [1]
    # valid output length after conv feature extractor
    out_len = encoder._get_feat_extract_output_lengths(in_len).item()

    if USE_AMP and DEVICE == "cuda":
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            out = encoder(input_values=input_values, attention_mask=attention_mask)
    else:
        out = encoder(input_values=input_values, attention_mask=attention_mask)

    hidden = out.last_hidden_state[0, :out_len, :]  # [L,H] only valid frames
    dur_sec = float(in_len.item()) / float(sr)
    stride_sec = dur_sec / max(1, int(out_len))
    return hidden, float(stride_sec)

def process_one_audio(video_id: str):
    mp3_path = AUDIO_DIR / f"{video_id}.mp3"
    if not mp3_path.exists():
        return None, {"video_id": video_id, "reason": "mp3_not_found"}

    out_npz  = OUT_EMB_DIR / f"{video_id}.npz"
    out_meta = OUT_EMB_DIR / f"{video_id}.json"

    if SKIP_IF_EXISTS and out_npz.exists() and out_meta.exists():
        try:
            meta = json.loads(out_meta.read_text(encoding="utf-8"))
            meta["video_id"] = video_id
            meta["npz_path"] = str(out_npz)
            return meta, None
        except Exception:
            return {"video_id": video_id, "npz_path": str(out_npz), "skipped": True}, None

    wav, sr = load_audio_mono_16k(mp3_path, TARGET_SR)
    wav = crop_wav(wav, sr, AUDIO_MAX_SEC, CROP_MODE)

    # fixed bins based on AUDIO_MAX_SEC
    timestamps = (np.arange(NUM_BINS, dtype=np.float32) * HOP_SEC)
    bin_start_sec = timestamps
    bin_end_sec = (timestamps + HOP_SEC).astype(np.float32)

    sums = torch.zeros((NUM_BINS, HIDDEN), device=DEVICE, dtype=torch.float32)
    cnts = torch.zeros((NUM_BINS,), device=DEVICE, dtype=torch.int32)

    # chunking inside cropped window
    chunk_samples   = int(CHUNK_SEC * sr)
    overlap_samples = int(OVERLAP_SEC * sr)
    step_samples    = max(1, chunk_samples - overlap_samples)

    wav_len = wav.numel()
    chunk_starts = list(range(0, wav_len, step_samples))

    for cs in chunk_starts:
        ce = min(wav_len, cs + chunk_samples)
        wav_chunk = wav[cs:ce]
        if wav_chunk.numel() < int(0.25 * sr):  # skip tiny tail
            continue

        hidden, stride_sec = wavlm_hidden_chunk(wav_chunk, sr)  # [L,H]
        L = hidden.shape[0]

        chunk_start_sec = float(cs) / float(sr)
        frame_times = chunk_start_sec + torch.arange(L, device=DEVICE, dtype=torch.float32) * float(stride_sec)

        bin_idx = torch.floor(frame_times / float(HOP_SEC)).long()
        valid = (bin_idx >= 0) & (bin_idx < NUM_BINS)
        if valid.any():
            ii = bin_idx[valid]
            hh = hidden[valid].float()
            sums.index_add_(0, ii, hh)
            cnts.index_add_(0, ii, torch.ones_like(ii, dtype=torch.int32))

    cnts_cpu = cnts.detach().cpu().numpy().astype(np.int32)
    valid_bins = (cnts_cpu > 0)

    emb = (sums / cnts.clamp_min(1).unsqueeze(1).float()).detach().cpu().numpy()

    if EMB_SAVE_DTYPE == "float16":
        emb_save = emb.astype(np.float16)
    else:
        emb_save = emb.astype(np.float32)

    meta = {
        "video_id": video_id,
        "mp3_path": str(mp3_path),
        "npz_path": str(out_npz),
        "encoder_dir": str(ENC_FT_DIR),
        "sr": int(sr),
        "audio_max_sec": float(AUDIO_MAX_SEC),
        "crop_mode": CROP_MODE,
        "fps": int(TARGET_FPS),
        "hop_sec": float(HOP_SEC),
        "num_bins": int(NUM_BINS),
        "hidden_size": int(HIDDEN),
        "chunk_sec": float(CHUNK_SEC),
        "overlap_sec": float(OVERLAP_SEC),
        "valid_bins": int(valid_bins.sum()),
        "save_dtype": EMB_SAVE_DTYPE,
    }

    np.savez_compressed(
        out_npz,
        timestamps_sec=bin_start_sec,
        bin_start_sec=bin_start_sec,
        bin_end_sec=bin_end_sec,
        valid=valid_bins.astype(np.bool_),
        counts=cnts_cpu,
        embeddings=emb_save,
    )
    out_meta.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")

    return meta, None

In [5]:
metas = []
failed = []

pbar = tqdm(all_ids, desc="Extract finetuned WavLM embeddings (fps=5)")
for i, vid in enumerate(pbar, start=1):
    try:
        meta, err = process_one_audio(vid)
        if err is not None:
            failed.append(err)
            tqdm.write(f"[{_ts()}] ERROR {vid}: {err['reason']}")
        else:
            metas.append(meta)
    except Exception as e:
        failed.append({"video_id": vid, "reason": repr(e)})
        tqdm.write(f"[{_ts()}] ERROR {vid}: {repr(e)}")
        tqdm.write(traceback.format_exc().rstrip())

    if i % 50 == 0:
        tqdm.write(f"[{_ts()}] progress {i}/{len(all_ids)} ok={len(metas)} failed={len(failed)}")

meta_df = pd.DataFrame(metas)
fail_df = pd.DataFrame(failed)

meta_df.to_parquet(INDEX_PATH, index=False)
if len(fail_df):
    fail_df.to_csv(OUT_EMB_DIR / "failed.csv", index=False)

print("DONE | ok:", len(meta_df), "failed:", len(fail_df))
print("Index:", INDEX_PATH)

Extract finetuned WavLM embeddings (fps=5):   0%|          | 0/12660 [00:00<?, ?it/s]

/home/danila/networks/.venv/lib/python3.12/site-packages/torch/nn/functional.py:6044: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


[17:55:46] progress 50/12660 ok=50 failed=0
[17:55:49] progress 100/12660 ok=100 failed=0
[17:55:53] progress 150/12660 ok=150 failed=0
[17:55:56] progress 200/12660 ok=200 failed=0
[17:55:59] progress 250/12660 ok=250 failed=0
[17:56:02] progress 300/12660 ok=300 failed=0
[17:56:05] progress 350/12660 ok=350 failed=0
[17:56:07] progress 400/12660 ok=400 failed=0
[17:56:10] progress 450/12660 ok=450 failed=0
[17:56:13] progress 500/12660 ok=500 failed=0
[17:56:16] progress 550/12660 ok=550 failed=0
[17:56:19] progress 600/12660 ok=600 failed=0
[17:56:22] progress 650/12660 ok=650 failed=0
[17:56:25] progress 700/12660 ok=700 failed=0
[17:56:28] progress 750/12660 ok=750 failed=0
[17:56:31] progress 800/12660 ok=800 failed=0
[17:56:34] progress 850/12660 ok=850 failed=0
[17:56:36] progress 900/12660 ok=900 failed=0
[17:56:39] progress 950/12660 ok=950 failed=0
[17:56:42] progress 1000/12660 ok=1000 failed=0
[17:56:45] progress 1050/12660 ok=1050 failed=0
[17:56:47] progress 1100/12660 o

In [ ]:
from transformers import WavLMModel, AutoFeatureExtractor
enc = WavLMModel.from_pretrained(ENC_FT_DIR)
fe  = AutoFeatureExtractor.from_pretrained(ENC_FT_DIR)